In [1]:
import random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [2]:
# =====================================================
# QUESTION 4 — MACHINE TRANSLATION
# Multi30k English → German
# Seq2Seq with Attention vs Pretrained Transformer
# Metrics: BLEU, METEOR, ChrF, BERTScore
# =====================================================

# 1. Install libraries
!pip install -q "datasets<4.0.0" transformers evaluate sacrebleu bert_score nltk

# =====================================================
# 2. Imports and seed
# =====================================================

import random
import re
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import nltk

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import evaluate

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

# =====================================================
# 3. Load Multi30k dataset
# =====================================================

dataset = load_dataset("bentrevett/multi30k")

print("\nDataset structure:")
print(dataset)

print("\nFirst example:")
print(dataset["train"][0])

# Multi30k has English-German sentence pairs.
# en = English source sentence
# de = German target sentence

# Use subsets for computational efficiency
TRAIN_SIZE = 5000
VAL_SIZE = 300

train_raw = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_raw = dataset["validation"].shuffle(seed=SEED).select(range(VAL_SIZE))

print("\nSubset sizes:")
print("Train:", len(train_raw))
print("Validation:", len(val_raw))

# =====================================================
# 4. Simple preprocessing and tokenization
# =====================================================

def simple_tokenize(text):
    """
    Simple tokenizer:
    - lowercase
    - separate punctuation
    - split by whitespace
    """
    text = text.lower().strip()
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

print("\nTokenization example:")
print("EN:", train_raw[0]["en"])
print(simple_tokenize(train_raw[0]["en"]))
print("DE:", train_raw[0]["de"])
print(simple_tokenize(train_raw[0]["de"]))

# =====================================================
# 5. Vocabulary construction
# =====================================================

class Vocab:
    def __init__(self, min_freq=2, max_size=10000):
        self.min_freq = min_freq
        self.max_size = max_size

        self.specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
        self.stoi = {tok: i for i, tok in enumerate(self.specials)}
        self.itos = {i: tok for tok, i in self.stoi.items()}

    def build(self, tokenized_sentences):
        freq = {}

        for sent in tokenized_sentences:
            for tok in sent:
                freq[tok] = freq.get(tok, 0) + 1

        sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)

        for tok, count in sorted_tokens:
            if count < self.min_freq:
                continue
            if len(self.stoi) >= self.max_size:
                break
            if tok not in self.stoi:
                idx = len(self.stoi)
                self.stoi[tok] = idx
                self.itos[idx] = tok

    def numericalize(self, tokens):
        return [self.stoi.get(tok, self.stoi["<unk>"]) for tok in tokens]

    def decode(self, ids):
        tokens = []
        for idx in ids:
            tok = self.itos.get(int(idx), "<unk>")
            if tok == "<eos>":
                break
            if tok not in ["<pad>", "<sos>", "<eos>"]:
                tokens.append(tok)
        return " ".join(tokens)

    def __len__(self):
        return len(self.stoi)


train_en_tokens = [simple_tokenize(ex["en"]) for ex in train_raw]
train_de_tokens = [simple_tokenize(ex["de"]) for ex in train_raw]

src_vocab = Vocab(min_freq=2, max_size=10000)
trg_vocab = Vocab(min_freq=2, max_size=10000)

src_vocab.build(train_en_tokens)
trg_vocab.build(train_de_tokens)

print("\nVocabulary sizes:")
print("Source vocab:", len(src_vocab))
print("Target vocab:", len(trg_vocab))

PAD_IDX = trg_vocab.stoi["<pad>"]
SOS_IDX = trg_vocab.stoi["<sos>"]
EOS_IDX = trg_vocab.stoi["<eos>"]

# =====================================================
# 6. Dataset and DataLoader for Seq2Seq
# =====================================================

MAX_LEN = 40

def encode_sentence(text, vocab, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    ids = [vocab.stoi["<sos>"]] + vocab.numericalize(tokens) + [vocab.stoi["<eos>"]]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids = ids + [vocab.stoi["<pad>"]] * (max_len - len(ids))

    return ids


class TranslationDataset(Dataset):
    def __init__(self, raw_data, src_vocab, trg_vocab):
        self.raw_data = raw_data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.raw_data)

    def __getitem__(self, idx):
        src_text = self.raw_data[idx]["en"]
        trg_text = self.raw_data[idx]["de"]

        src_ids = encode_sentence(src_text, self.src_vocab, MAX_LEN)
        trg_ids = encode_sentence(trg_text, self.trg_vocab, MAX_LEN)

        return {
            "src_ids": torch.tensor(src_ids, dtype=torch.long),
            "trg_ids": torch.tensor(trg_ids, dtype=torch.long),
            "src_text": src_text,
            "trg_text": trg_text
        }


train_dataset = TranslationDataset(train_raw, src_vocab, trg_vocab)
val_dataset = TranslationDataset(val_raw, src_vocab, trg_vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

batch = next(iter(train_loader))
print("\nBatch shapes:")
print(batch["src_ids"].shape)
print(batch["trg_ids"].shape)

# =====================================================
# 7. Seq2Seq with Attention Model
# =====================================================

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=src_vocab.stoi["<pad>"])
        self.rnn = nn.GRU(
            emb_dim,
            enc_hid_dim,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))

        outputs, hidden = self.rnn(embedded)

        # hidden shape: [2, batch, enc_hid_dim]
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]

        hidden = torch.tanh(self.fc(torch.cat((hidden_forward, hidden_backward), dim=1)))

        return outputs, hidden


class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()

        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, src_mask):
        # hidden: [batch, dec_hid_dim]
        # encoder_outputs: [batch, src_len, enc_hid_dim * 2]

        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))

        attention = self.v(energy).squeeze(2)

        attention = attention.masked_fill(src_mask == 0, -1e10)

        return torch.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=trg_vocab.stoi["<pad>"])

        self.rnn = nn.GRU(
            (enc_hid_dim * 2) + emb_dim,
            dec_hid_dim,
            batch_first=True
        )

        self.fc_out = nn.Linear(
            (enc_hid_dim * 2) + dec_hid_dim + emb_dim,
            output_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs, src_mask):
        # input_token: [batch]
        input_token = input_token.unsqueeze(1)

        embedded = self.dropout(self.embedding(input_token))

        attention_weights = self.attention(hidden, encoder_outputs, src_mask)
        attention_weights = attention_weights.unsqueeze(1)

        weighted = torch.bmm(attention_weights, encoder_outputs)

        rnn_input = torch.cat((embedded, weighted), dim=2)

        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))

        output = output.squeeze(1)
        hidden = hidden.squeeze(0)
        embedded = embedded.squeeze(1)
        weighted = weighted.squeeze(1)

        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))

        return prediction, hidden, attention_weights.squeeze(1)


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.device = device

    def create_src_mask(self, src):
        return (src != self.src_pad_idx)

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden = self.encoder(src)

        src_mask = self.create_src_mask(src)

        input_token = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(
                input_token,
                hidden,
                encoder_outputs,
                src_mask
            )

            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input_token = trg[:, t] if teacher_force else top1

        return outputs


INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256
DROPOUT = 0.3

attention = Attention(ENC_HID_DIM, DEC_HID_DIM)
encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DROPOUT)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DROPOUT, attention)

seq2seq_model = Seq2Seq(
    encoder,
    decoder,
    src_vocab.stoi["<pad>"],
    device
).to(device)

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.stoi["<pad>"])

print("\nSeq2Seq model created.")

# =====================================================
# 8. Train Seq2Seq with Attention
# =====================================================

def train_epoch(model, dataloader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    for batch in dataloader:
        src = batch["src_ids"].to(device)
        trg = batch["trg_ids"].to(device)

        optimizer.zero_grad()

        output = model(src, trg, teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]

        output = output[:, 1:, :].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


def evaluate_epoch(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            src = batch["src_ids"].to(device)
            trg = batch["trg_ids"].to(device)

            output = model(src, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]

            output = output[:, 1:, :].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


N_EPOCHS = 5

print("\nTraining Seq2Seq with Attention...")

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(seq2seq_model, train_loader, optimizer, criterion)
    val_loss = evaluate_epoch(seq2seq_model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print("-" * 40)

# =====================================================
# 9. Seq2Seq Translation Function
# =====================================================

def translate_seq2seq(sentence, model, src_vocab, trg_vocab, max_len=40):
    model.eval()

    src_ids = encode_sentence(sentence, src_vocab, MAX_LEN)
    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    src_mask = model.create_src_mask(src_tensor)

    input_token = torch.tensor([trg_vocab.stoi["<sos>"]], dtype=torch.long).to(device)

    generated_ids = []

    attentions = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, attention_weights = model.decoder(
                input_token,
                hidden,
                encoder_outputs,
                src_mask
            )

        top1 = output.argmax(1).item()

        if top1 == trg_vocab.stoi["<eos>"]:
            break

        generated_ids.append(top1)
        attentions.append(attention_weights.cpu().numpy())

        input_token = torch.tensor([top1], dtype=torch.long).to(device)

    return trg_vocab.decode(generated_ids)


# =====================================================
# 10. Pretrained Transformer Translation Model
# =====================================================

# Helsinki-NLP opus-mt-en-de is a pretrained Transformer-based MarianMT model.
transformer_model_name = "Helsinki-NLP/opus-mt-en-de"

print("\nLoading pretrained Transformer model...")
transformer_tokenizer = AutoTokenizer.from_pretrained(transformer_model_name)
transformer_model = AutoModelForSeq2SeqLM.from_pretrained(transformer_model_name).to(device)

def translate_transformer(sentence):
    inputs = transformer_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        generated_ids = transformer_model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    return transformer_tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

# =====================================================
# 11. Generate translations on validation subset
# =====================================================

eval_size = 100
eval_data = val_raw.select(range(eval_size))

sources = eval_data["en"]
references = eval_data["de"]

seq2seq_outputs = []
transformer_outputs = []

print("\nGenerating translations...")

for i, src_sentence in enumerate(sources):
    print(f"Translating {i+1}/{eval_size}")

    seq2seq_translation = translate_seq2seq(
        src_sentence,
        seq2seq_model,
        src_vocab,
        trg_vocab
    )

    transformer_translation = translate_transformer(src_sentence)

    seq2seq_outputs.append(seq2seq_translation)
    transformer_outputs.append(transformer_translation)

print("\nExample translations:")
print("Source:", sources[0])
print("Reference:", references[0])
print("Seq2Seq:", seq2seq_outputs[0])
print("Transformer:", transformer_outputs[0])

# =====================================================
# 12. Evaluation Metrics
# =====================================================

sacrebleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
chrf_metric = evaluate.load("chrf")
bertscore_metric = evaluate.load("bertscore")

def evaluate_translations(predictions, references, method_name):
    print(f"\nEvaluating {method_name}...")

    bleu = sacrebleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    meteor = meteor_metric.compute(
        predictions=predictions,
        references=references
    )

    chrf = chrf_metric.compute(
        predictions=predictions,
        references=references
    )

    bertscore = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="de"
    )

    avg_bertscore_f1 = float(np.mean(bertscore["f1"]))

    return {
        "Model": method_name,
        "BLEU": bleu["score"],
        "METEOR": meteor["meteor"],
        "ChrF": chrf["score"],
        "BERTScore-F1": avg_bertscore_f1
    }


seq2seq_results = evaluate_translations(
    seq2seq_outputs,
    references,
    "Seq2Seq + Attention"
)

transformer_results = evaluate_translations(
    transformer_outputs,
    references,
    "Pretrained Transformer"
)

results_df_q4 = pd.DataFrame([seq2seq_results, transformer_results])

print("\n=========================")
print("FINAL Q4 RESULTS TABLE")
print("=========================")
print(results_df_q4)

# =====================================================
# 13. Qualitative Example
# =====================================================

print("\n=========================")
print("QUALITATIVE EXAMPLES")
print("=========================")

for i in range(3):
    print("=" * 100)
    print(f"Example {i+1}")
    print("=" * 100)

    print("\nSOURCE ENGLISH:")
    print(sources[i])

    print("\nREFERENCE GERMAN:")
    print(references[i])

    print("\nSEQ2SEQ + ATTENTION OUTPUT:")
    print(seq2seq_outputs[i])

    print("\nPRETRAINED TRANSFORMER OUTPUT:")
    print(transformer_outputs[i])

# =====================================================
# 14. Save outputs
# =====================================================

qualitative_data = []

for i in range(eval_size):
    qualitative_data.append({
        "source_english": sources[i],
        "reference_german": references[i],
        "seq2seq_attention_output": seq2seq_outputs[i],
        "pretrained_transformer_output": transformer_outputs[i]
    })

qualitative_df_q4 = pd.DataFrame(qualitative_data)

results_df_q4.to_csv("q4_machine_translation_results.csv", index=False)
qualitative_df_q4.to_csv("q4_translation_examples.csv", index=False)

print("\nSaved files:")
print("q4_machine_translation_results.csv")
print("q4_translation_examples.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.8 MB/s eta 0:00:00
Device: cuda


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluating Pretrained Transformer...

FINAL Q4 RESULTS TABLE
                    Model       BLEU    METEOR       ChrF  BERTScore-F1
0     Seq2Seq + Attention   1.645356  0.396140  29.297068      0.685307
1  Pretrained Transformer  36.511180  0.646675  63.652023      0.894514

QUALITATIVE EXAMPLES
Example 1

SOURCE ENGLISH:
2 girls playing volleyball, one striking the ball.

REFERENCE GERMAN:
Zwei Mädchen spielen Volleyball, wobei das eine den Ball schlägt.

SEQ2SEQ + ATTENTION OUTPUT:
<unk> mädchen spielen <unk> <unk> den ball .

PRETRAINED TRANSFORMER OUTPUT:
2 Mädchen spielen Volleyball, eine schlägt den Ball.
Example 2

SOURCE ENGLISH:
A man and woman are walking down a waterside path toward a suspension bridge.

REFERENCE GERMAN:
Ein Mann und eine Frau gehen einen Uferweg entlang auf eine Hängebrücke zu.

SEQ2SEQ + ATTENTION OUTPUT:
ein mann und eine frau gehen gehen einen <unk> entlang , die eine <unk> entlang .

PRETRAINED TRANSFORMER OUTPUT:
Ein Mann und eine Frau gehen einen 